# 第 1 周：职位帖分析

## 练习目标（理念）

抓取招聘页面的纯文本，再用 **Groq** 上的 Llama 模型做结构化职位分析与投递建议（Apply / Skip）。

- **输入**：职位页 URL（网页 HTML）
- **中间产物**：去噪后的纯文本
- **输出**：职位标题、技能、经验、薪资线索、是否投递的建议

## 和本课 Day 1 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| HTTP 抓取 + HTML 解析 | `requests.get` + BeautifulSoup |
| Chat Completions（`messages`） | system 定「怎么分析」，user 放职位正文 |
| 环境变量装密钥 | `.env` 里的 `GROQ_API_KEY` |
| 云端推理 API | Groq SDK + `llama-3.1-8b-instant` |

## 怎么跑

1. 从上到下依次运行每个单元格（Shift+Enter）
2. 准备好 `.env`：至少有 `GROQ_API_KEY`（Groq 密钥通常以 `gsk_` 开头）
3. 在最后一格把 `url = ""` 改成真实职位页链接，再运行分析

> 说明：本笔记本调用 `client.chat.completions.create`；若本机尚未创建 `client = Groq(...)`，需按原作者意图补上（此处不改可执行逻辑）。


In [ ]:
# ========== 导入：把后面要用的工具箱搬进来 ==========

# 导入标准库 os：读环境变量（Environment Variables），例如 GROQ_API_KEY
import os
# 导入标准库 requests：用 HTTP GET 下载职位页 HTML
import requests
# 从 bs4 导入 BeautifulSoup：把 HTML 解析成可遍历的文档树，方便抽纯文本
from bs4 import BeautifulSoup
# 从 dotenv 导入 load_dotenv：把 .env 里的密钥读进环境变量，避免把密钥写进代码
from dotenv import load_dotenv
# 从 groq 导入 Groq 客户端类：调用 Groq 云端的 Chat Completions API
from groq import Groq


In [ ]:
# ========== 环境检查：确认 GROQ_API_KEY 已正确加载 ==========

# 再次导入 os / load_dotenv（与上一格重复导入无害；保持原逻辑不重构）
import os
from dotenv import load_dotenv

# 加载 .env：override=True 表示用文件里的值覆盖进程里已有的同名环境变量
load_dotenv(override=True)
# 从环境变量取出 Groq API Key（字符串必须原样保留，不能改名）
api_key = os.getenv('GROQ_API_KEY')

# 分支校验：缺密钥 / 格式不对 / 首尾多余空格 —— print 文案保持英文（影响行为的字符串不翻译）
if not api_key:
    print("No API key found")
elif not api_key.startswith("gsk_"):
    # Groq 密钥惯例以 gsk_ 开头；不匹配多半是粘贴错了
    print("Invalid Groq API key format")
elif api_key.strip() != api_key:
    # strip 后与原串不同 → 说明有前导/尾随空白，后续请求可能失败
    print("API key has extra spaces")
else:
    print("Groq API key looks good!")


## 抓取器（Scraper）

下面定义 `fetch_job_text(url)`：下载职位页 HTML，去掉 `script` / `style` 噪音，压成单行纯文本，供后续 prompt 使用。


In [ ]:
# ========== 抓取：URL → 去噪纯文本 ==========

def fetch_job_text(url):
    # 抓取职位页纯文本：输入 URL，返回压缩空白后的字符串
    # User-Agent：模拟浏览器；部分站点对缺省 UA 的脚本请求更不友好
    headers = {"User-Agent": "Mozilla/5.0"}

    # GET 下载整页 HTML；response.text 是解码后的字符串
    response = requests.get(url, headers=headers)
    # 用 html.parser 建 DOM；也可换成 lxml，但这里保持原参数
    soup = BeautifulSoup(response.text, "html.parser")

    # 去掉脚本/样式等噪音：decompose 会从树里删除节点及其子树
    for tag in soup(["script", "style"]):
        tag.decompose()

    # get_text：把可见文本抽出来；separator=" " 在块之间插空格，避免词粘连
    text = soup.get_text(separator=" ")

    # split()+join：折叠连续空白/换行，得到更紧凑的单行文本，利于塞进 prompt
    return " ".join(text.split())


In [ ]:
# ========== system prompt：定模型角色与输出结构（发给模型的英文指令不翻译）==========

# 三引号多行字符串：整段作为 Chat Completions 的 system message content
system_prompt = """
You are an expert AI Job Analyst.

Your job:
- Analyze job posts clearly and structured
- Extract useful insights
- Be concise and practical

Return format:
1. Job Title
2. Required Skills
3. Experience Level
4. Salary clues (if any)
5. Recommendation (Apply or Skip with reason)

Rules:
- Be direct
- No fluff
- Use bullet points when needed
"""


In [ ]:
# ========== user prompt 模板：真正装职位正文的那一侧 ==========

# 注意：模板里有 {job_text} 占位，但后面 analyze_job 用的是字符串拼接而不是 .format()
# 因此 {job_text} 会原样出现在发给模型的文本前缀里；保持原逻辑不改
user_prompt = """Analyze this job post: {job_text}"""


In [ ]:
# ========== 分析：把职位纯文本交给 Groq 上的 Llama ==========

def analyze_job(job_text):
    # 调用模型分析职位帖：Chat Completions 一次请求，非流式
    # client 需事先是 Groq(...) 实例；model id 字符串必须与 Groq 可用模型一致
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            # system：角色 + 输出格式（上面定义的 system_prompt）
            {"role": "system", "content": system_prompt},
            # user：模板前缀 + 职位正文；括号仅分组，等价于直接拼接
            {"role": "user", "content": user_prompt + (job_text)}
        ]
    )

    # choices[0].message.content：取第一条候选回答的正文
    return response.choices[0].message.content


In [ ]:
# ========== 主流程：抓取 → 分析 → 打印 ==========

# 把真实职位页 URL 填进这个字符串（当前为空：需你自己改）
url = ""
# 进度提示保持英文（print 给人看也可中文化，但原串影响不大，按「可运行英文」保留）
print("\nScraping job post...\n")
# 下载并清洗职位页文本
job_text = fetch_job_text(url)

print("Analyzing with AI...\n")
# 把纯文本交给模型，拿到结构化分析
result = analyze_job(job_text)

# 打印模型返回的全文（标题/技能/建议等）
print(result)
